# PPO Beam Tracker — Enriched Observation Training

Changes vs. original notebook:
- **Observation**: 22-dim instead of 11 (`blocked_hist`, `outage_hist`, `beam_trend` added)
- **Reward**: potential-based SNR shaping + `outage_weight=3.0`
- **PPO hyperparams**: wider MLP, entropy bonus, longer rollout horizon
- **Training output**: one progress line per 10% of training, no SB3 spam
- **Baselines**: random + neighbor (left/right ±1) evaluated on same seeds

In [ ]:
# Cell 0 — Repo root detection
import os, sys
from pathlib import Path

def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / "beam_tracker_rl").is_dir() and (p / "neighbor_baseline.py").exists():
            return p
    raise RuntimeError(
        "Could not find repo root. Start Jupyter from inside beam_tracker_RL."
    )

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

In [ ]:
# Cell 1 — Imports and smoke test
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

import beam_tracker_rl
from beam_tracker_rl import ENV_ID
from beam_tracker_rl.sim import MovementConfig, RewardConfig

SCENARIO       = "single_occluder"
MOVEMENT_CONFIG = MovementConfig(model="stochastic")
# outage_weight=3.0 matches the updated sim.py default.
# If you want to override: RewardConfig(outage_weight=3.0)
REWARD_CONFIG  = RewardConfig()

# Quick sanity check
env = gym.make(ENV_ID, scenario_name=SCENARIO,
               movement_config=MOVEMENT_CONFIG,
               reward_config=REWARD_CONFIG)
obs, info = env.reset(seed=0)
env.close()

print(f"Env: {ENV_ID}  |  obs shape: {obs.shape}  |  n_actions: {env.action_space.n}")
print(f"Initial SNR: {info['snr_db']:.1f} dB  |  beam: {info['selected_beam_idx']}  "
      f"optimal: {info['optimal_beam_idx']}")

## Training configuration

- Set `RESUME_FROM` to a checkpoint path to continue an interrupted run.
- Training prints one progress line per 10 % of total steps — no SB3 noise.

In [ ]:
# Cell 2 — Training configuration
TOTAL_TIMESTEPS  = 1_000_000
CHECKPOINT_EVERY = 100_000

MODEL_DIR      = REPO_ROOT / "models"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
FINAL_DIR      = MODEL_DIR / "final"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Set to e.g. "models/checkpoints/ppo_single_occluder_500000_steps.zip" to resume.
RESUME_FROM = None

print(f"Total timesteps : {TOTAL_TIMESTEPS:,}")
print(f"Checkpoint every: {CHECKPOINT_EVERY:,}")
print(f"Resume from     : {RESUME_FROM}")

In [ ]:
# Cell 3 — Build or resume PPO model
train_env = gym.make(ENV_ID, scenario_name=SCENARIO,
                     movement_config=MOVEMENT_CONFIG,
                     reward_config=REWARD_CONFIG)
train_env = Monitor(train_env)

checkpoint_cb = CheckpointCallback(
    save_freq=CHECKPOINT_EVERY,
    save_path=str(CHECKPOINT_DIR),
    name_prefix=f"ppo_{SCENARIO}",
    save_replay_buffer=False,
    save_vecnormalize=False,
)

if RESUME_FROM:
    print(f"Resuming from: {RESUME_FROM}")
    model = PPO.load(RESUME_FROM, env=train_env)
else:
    model = PPO(
        "MlpPolicy",
        train_env,
        verbose=0,                          # suppress all SB3 output
        learning_rate=1e-4,
        n_steps=2048,                       # longer rollout — sees full occlusion events
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        ent_coef=0.01,                      # entropy bonus: encourages beam exploration
        clip_range=0.2,
        policy_kwargs=dict(net_arch=[128, 128]),  # wider than SB3 default [64, 64]
        tensorboard_log=str(REPO_ROOT / "runs"),
        device="cpu",
    )
    print("New PPO model created.")

print(f"Policy network: {model.policy}")

In [ ]:
# Cell 4 — Train with a single progress-line callback

class QuietProgressCallback(BaseCallback):
    """Prints one line every `report_every` timesteps. No other output."""

    def __init__(self, total_timesteps: int, report_every: int = 100_000):
        super().__init__(verbose=0)
        self.total = total_timesteps
        self.report_every = report_every
        self._next_report = report_every

    def _on_step(self) -> bool:
        t = self.num_timesteps
        if t >= self._next_report:
            pct = 100.0 * t / self.total
            # Pull mean episode reward from Monitor if available
            ep_rews = self.model.ep_info_buffer
            if ep_rews:
                mean_rew = np.mean([ep["r"] for ep in ep_rews])
                print(f"  [{pct:5.1f}%]  step {t:>9,}  mean_ep_reward={mean_rew:+.2f}")
            else:
                print(f"  [{pct:5.1f}%]  step {t:>9,}")
            self._next_report += self.report_every
        return True


from stable_baselines3.common.callbacks import CallbackList

callbacks = CallbackList([checkpoint_cb, QuietProgressCallback(TOTAL_TIMESTEPS)])

print(f"Training for {TOTAL_TIMESTEPS:,} timesteps ...")
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callbacks, log_interval=None)

final_path = FINAL_DIR / f"ppo_{SCENARIO}_{TOTAL_TIMESTEPS}_steps"
model.save(str(final_path))
print(f"\nDone. Saved → {final_path}.zip")

## Evaluation

Three policies compared on the same episode seeds:
1. **Random** — uniform random beam selection
2. **Neighbor** — greedy ±1 beam search (the baseline PPO was losing to)
3. **PPO** — trained policy

In [ ]:
# Cell 5 — Shared evaluation harness

def _make_env():
    return gym.make(ENV_ID, scenario_name=SCENARIO,
                    movement_config=MOVEMENT_CONFIG,
                    reward_config=REWARD_CONFIG)


def run_episode(policy_fn, seed: int) -> dict:
    """Roll out one episode with policy_fn(obs) -> action. Returns summary dict."""
    env = _make_env()
    obs, info = env.reset(seed=seed)
    ep_reward, snrs, outages, matches = 0.0, [], [], []
    done = False
    while not done:
        action = policy_fn(obs, env)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        ep_reward += float(reward)
        snrs.append(float(info["snr_db"]))
        outages.append(float(info["outage"]))
        matches.append(float(info["selected_beam_idx"] == info["optimal_beam_idx"]))
    env.close()
    return dict(
        reward=ep_reward,
        mean_snr=float(np.mean(snrs)),
        outage_rate=float(np.mean(outages)),
        beam_match=float(np.mean(matches)),
    )


def evaluate(policy_fn, n_episodes=30, seed_start=1000) -> dict:
    results = [run_episode(policy_fn, seed_start + ep) for ep in range(n_episodes)]
    out = {}
    for key in results[0]:
        vals = [r[key] for r in results]
        out[f"{key}_mean"] = float(np.mean(vals))
        out[f"{key}_std"]  = float(np.std(vals))
    return out


# ── Policy functions ────────────────────────────────────────────────────────

def random_policy(obs, env):
    return env.action_space.sample()


def neighbor_policy(obs, env):
    """Greedy ±1 beam search: probe left, right, stay — pick highest SNR.
    Reads current action from env state (same information the baseline uses).
    """
    current = env.unwrapped.current_action
    n = env.action_space.n
    candidates = {current}  # always include staying
    if current > 0:
        candidates.add(current - 1)
    if current < n - 1:
        candidates.add(current + 1)

    best_action, best_snr = current, -999.0
    ue_state  = env.unwrapped.ue_state
    scenario  = env.unwrapped.scenario
    codebook  = env.unwrapped.codebook
    channel   = env.unwrapped.channel_config
    reward_cfg = env.unwrapped.reward_config

    from beam_tracker_rl.sim import (
        beam_angle_from_action, true_angle_deg, angle_error_deg,
        euclidean_distance, is_blocked, compute_snr,
    )
    ue_xy    = (ue_state.x, ue_state.y)
    distance = euclidean_distance(scenario.bs_xy, ue_xy)
    blocked  = is_blocked(scenario.bs_xy, ue_xy, scenario.obstacles)

    for action in candidates:
        beam_deg = beam_angle_from_action(action, codebook)
        theta    = true_angle_deg(scenario.bs_xy, ue_xy)
        err      = angle_error_deg(theta, beam_deg)
        snr      = compute_snr(distance, err, blocked, channel)["snr_db"]
        if snr > best_snr:
            best_snr, best_action = snr, action
    return best_action


def ppo_policy(obs, env):
    action, _ = model.predict(obs, deterministic=True)
    return int(action)


print("Evaluation functions ready.")

In [ ]:
# Cell 6 — Run all three evaluations
N_EVAL = 30
SEED0  = 5000   # disjoint from training seeds

print("Evaluating random  ...", end=" ", flush=True)
random_metrics   = evaluate(random_policy,   N_EVAL, SEED0)
print("done")

print("Evaluating neighbor...", end=" ", flush=True)
neighbor_metrics = evaluate(neighbor_policy, N_EVAL, SEED0)
print("done")

print("Evaluating PPO     ...", end=" ", flush=True)
ppo_metrics      = evaluate(ppo_policy,      N_EVAL, SEED0)
print("done")

# ── Summary table ────────────────────────────────────────────────────────────
rows = []
for name, m in [("Random", random_metrics), ("Neighbor", neighbor_metrics), ("PPO", ppo_metrics)]:
    rows.append({
        "Policy"        : name,
        "Reward"        : f"{m['reward_mean']:+.1f} ± {m['reward_std']:.1f}",
        "Mean SNR (dB)" : f"{m['mean_snr_mean']:.1f} ± {m['mean_snr_std']:.1f}",
        "Outage rate"   : f"{m['outage_rate_mean']:.3f} ± {m['outage_rate_std']:.3f}",
        "Beam match"    : f"{m['beam_match_mean']:.3f} ± {m['beam_match_std']:.3f}",
    })

pd.DataFrame(rows).set_index("Policy")

## Single-episode rollout plots

Four subplots: SNR over time, beam selection vs. oracle, blockage/outage indicators, UE trajectory.

In [ ]:
# Cell 7 — Rollout collector

def collect_rollout(policy_fn, seed: int = 3000) -> pd.DataFrame:
    env = _make_env()
    obs, info = env.reset(seed=seed)
    rows = []
    done = False
    while not done:
        action = policy_fn(obs, env)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        ue_x, ue_y = info["ue_xy"]
        rows.append({
            "t"                : info["t"],
            "reward"           : reward,
            "snr_db"           : info["snr_db"],
            "outage"           : int(info["outage"]),
            "blocked"          : int(info["blocked"]),
            "selected_beam_idx": info["selected_beam_idx"],
            "optimal_beam_idx" : info["optimal_beam_idx"],
            "selected_beam_deg": info["selected_beam_deg"],
            "optimal_beam_deg" : info["optimal_beam_deg"],
            "ue_x"             : ue_x,
            "ue_y"             : ue_y,
        })
    env.close()
    return pd.DataFrame(rows)


def plot_rollout(df: pd.DataFrame, title: str, save_path=None):
    fig, axes = plt.subplots(4, 1, figsize=(12, 14), constrained_layout=True)
    fig.suptitle(title, fontsize=14)

    # — SNR —
    ax = axes[0]
    ax.plot(df["t"], df["snr_db"], color="steelblue", lw=1.5)
    ax.axhline(5.0, color="red", lw=1, linestyle="--", label="outage thresh (5 dB)")
    ax.fill_between(df["t"], df["snr_db"], 5.0,
                    where=(df["snr_db"] < 5.0), alpha=0.25, color="red")
    ax.set_ylabel("SNR (dB)")
    ax.set_title("SNR over time")
    ax.legend(fontsize=8)

    # — Beam selection —
    ax = axes[1]
    ax.plot(df["t"], df["selected_beam_idx"], label="Selected", color="steelblue", lw=1.5)
    ax.plot(df["t"], df["optimal_beam_idx"],  label="Oracle",   color="orange",
            lw=1.5, linestyle="--")
    ax.set_ylabel("Beam index")
    ax.set_title("Selected beam vs oracle nearest beam")
    ax.legend(fontsize=8)

    # — Blockage / outage —
    ax = axes[2]
    ax.fill_between(df["t"], df["blocked"], alpha=0.4, color="goldenrod", label="Blocked")
    ax.fill_between(df["t"], df["outage"],  alpha=0.5, color="red",      label="Outage")
    ax.set_ylim(-0.05, 1.15)
    ax.set_ylabel("Indicator")
    ax.set_title("Blockage and outage")
    ax.legend(fontsize=8)

    # — UE trajectory —
    ax = axes[3]
    sc = ax.scatter(df["ue_x"], df["ue_y"], c=df["snr_db"],
                    cmap="RdYlGn", s=6, vmin=0, vmax=30)
    plt.colorbar(sc, ax=ax, label="SNR (dB)")
    ax.scatter(*df[["ue_x", "ue_y"]].iloc[0],  marker="^", s=80,
               color="green",  zorder=5, label="Start")
    ax.scatter(*df[["ue_x", "ue_y"]].iloc[-1], marker="s", s=80,
               color="black",  zorder=5, label="End")
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
    ax.set_title("UE trajectory (colour = SNR)")
    ax.legend(fontsize=8)

    for ax in axes[:3]:
        ax.set_xlabel("Step")

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


print("Rollout helpers ready.")

In [ ]:
# Cell 8 — Plot rollouts for all three policies on the same seed
ROLLOUT_SEED = 3000
plot_dir = REPO_ROOT / "models"

for name, fn in [("PPO", ppo_policy), ("Neighbor", neighbor_policy), ("Random", random_policy)]:
    df = collect_rollout(fn, seed=ROLLOUT_SEED)
    save = plot_dir / f"rollout_{name.lower()}_{ROLLOUT_SEED}.png"
    plot_rollout(df, title=f"{name} — seed {ROLLOUT_SEED}", save_path=save)

In [ ]:
# Cell 9 — List saved checkpoints and final models
print("Checkpoints:")
for p in sorted(CHECKPOINT_DIR.glob("*.zip")):
    print(" ", p.name)

print("\nFinal models:")
for p in sorted(FINAL_DIR.glob("*.zip")):
    print(" ", p.name)